# Random Forest Classification
Each small decision tree predicts a class. The forest chooses the class with the most votes. Fixed bootstrap samples keep the example simple and repeatable.

In [ ]:
sizes = [950, 1200, 1450, 1700, 2000]
fast_sale = ['No', 'Yes', 'Yes', 'Yes', 'Yes']

def majority_vote(labels):
    yes_count = 0
    no_count = 0
    for label in labels:
        if label == 'Yes':
            yes_count = yes_count + 1
        else:
            no_count = no_count + 1

    if yes_count >= no_count:
        return 'Yes'
    return 'No'

def gini_score(labels):
    yes_count = 0
    no_count = 0
    for label in labels:
        if label == 'Yes':
            yes_count = yes_count + 1
        else:
            no_count = no_count + 1

    total = len(labels)
    yes_probability = yes_count / total
    no_probability = no_count / total
    score = 1 - yes_probability ** 2 - no_probability ** 2
    return score * total

## Train one small classification tree

In [ ]:
def train_classification_stump(sample_sizes, sample_labels):
    unique_sizes = sorted(set(sample_sizes))
    best_tree = None
    best_score = float('inf')

    for i in range(len(unique_sizes) - 1):
        threshold = (unique_sizes[i] + unique_sizes[i + 1]) / 2
        left_labels = []
        right_labels = []

        for j in range(len(sample_sizes)):
            if sample_sizes[j] < threshold:
                left_labels.append(sample_labels[j])
            else:
                right_labels.append(sample_labels[j])

        score = gini_score(left_labels) + gini_score(right_labels)
        if score < best_score:
            best_score = score
            best_tree = {
                'threshold': threshold,
                'left_answer': majority_vote(left_labels),
                'right_answer': majority_vote(right_labels)
            }
    return best_tree

def tree_prediction(tree, size):
    if size < tree['threshold']:
        return tree['left_answer']
    return tree['right_answer']

## Build three trees with fixed bootstrap samples

In [ ]:
bootstrap_indices = [
    [0, 1, 1, 3, 4],
    [0, 0, 2, 3, 4],
    [1, 2, 2, 3, 4]
]

forest = []
for tree_number in range(len(bootstrap_indices)):
    sample_sizes = []
    sample_labels = []
    selected_indices = bootstrap_indices[tree_number]

    for index in selected_indices:
        sample_sizes.append(sizes[index])
        sample_labels.append(fast_sale[index])

    tree = train_classification_stump(sample_sizes, sample_labels)
    forest.append(tree)
    print('Tree', tree_number + 1, tree)

## Use majority voting for the final answer

In [ ]:
test_sizes = [1280, 460]
for test_size in test_sizes:
    votes = []
    for tree in forest:
        votes.append(tree_prediction(tree, test_size))

    final_answer = majority_vote(votes)
    print('Size:', test_size)
    print('Tree votes:', votes)
    print('Fast sale:', final_answer)